In [12]:

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

from docforge.models.unet import UNet
from docforge.training.dataset import DocTamperDataset
from docforge.training.metrics import calculate_metrics

# Project paths
PROJECT_ROOT = Path.cwd().parent

split_df = pd.read_csv(
    PROJECT_ROOT / "data/splits/large_dataset_split.csv"
)

lmdb_path = PROJECT_ROOT / "data/raw/large"
checkpoint_path = PROJECT_ROOT / "checkpoints/unet_smoke_test.pth"

# Use validation data
val_df = split_df[split_df["split"] == "validation"].copy()

val_dataset = DocTamperDataset(val_df, lmdb_path)

# Only one batch for this test
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

# Select device
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load model and checkpoint
model = UNet(in_channels=3, out_channels=1).to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Calculate metrics
images, masks = next(iter(val_loader))

images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    outputs = model(images)

results = calculate_metrics(outputs, masks)

print("Device:", device)
print("Evaluation images:", len(images))
print("\nSegmentation metrics:")

for metric, value in results.items():
    print(f"{metric.capitalize()}: {value:.4f}")


Device: mps
Evaluation images: 4

Segmentation metrics:
Dice: 0.0559
Iou: 0.0288
Precision: 0.0288
Recall: 1.0000


In [3]:
print("Project root:", PROJECT_ROOT)
print("Split CSV exists:", 
      (PROJECT_ROOT / "data/splits/large_dataset_split.csv").exists())
print("Checkpoint exists:",
      (PROJECT_ROOT / "checkpoints/unet_smoke_test.pth").exists())

Project root: /Users/thinyadanaroo/sch_projects/PROM02/PROM02_240291316/doc-forgery-detection
Split CSV exists: True
Checkpoint exists: True


In [4]:
# Before training, inspect prediction behaviour

# Calculate predicted and actual tampered pixel percentages

probabilities = torch.sigmoid(outputs)
predictions = (probabilities >= 0.5).float()

for i in range(len(images)):
    predicted_percentage = predictions[i].mean().item() * 100
    actual_percentage = masks[i].mean().item() * 100

    print(f"Image {i + 1}")
    print(f"Predicted tampered pixels: {predicted_percentage:.2f}%")
    print(f"Actual tampered pixels:    {actual_percentage:.2f}%")
    print("-" * 35)

Image 1
Predicted tampered pixels: 100.00%
Actual tampered pixels:    9.60%
-----------------------------------
Image 2
Predicted tampered pixels: 100.00%
Actual tampered pixels:    0.48%
-----------------------------------
Image 3
Predicted tampered pixels: 100.00%
Actual tampered pixels:    0.44%
-----------------------------------
Image 4
Predicted tampered pixels: 100.00%
Actual tampered pixels:    0.98%
-----------------------------------


In [5]:
# inspect the models' raw probabilities 

for i in range(len(images)):
    probs = probabilities[i]

    print(f"Image {i + 1}")
    print(f"Minimum probability: {probs.min().item():.4f}")
    print(f"Maximum probability: {probs.max().item():.4f}")
    print(f"Mean probability:    {probs.mean().item():.4f}")
    print(f"Median probability:  {probs.median().item():.4f}")
    print("-" * 40)


Image 1
Minimum probability: 0.5067
Maximum probability: 0.5262
Mean probability:    0.5105
Median probability:  0.5103
----------------------------------------
Image 2
Minimum probability: 0.5054
Maximum probability: 0.5275
Mean probability:    0.5107
Median probability:  0.5104
----------------------------------------
Image 3
Minimum probability: 0.5062
Maximum probability: 0.5272
Mean probability:    0.5109
Median probability:  0.5108
----------------------------------------
Image 4
Minimum probability: 0.5060
Maximum probability: 0.5289
Mean probability:    0.5148
Median probability:  0.5147
----------------------------------------


In [6]:
for i in range(len(images)):
    logits_i = outputs[i, 0]

    print(f"Image {i + 1}")
    print(f"Logit minimum: {logits_i.min().item():.4f}")
    print(f"Logit maximum: {logits_i.max().item():.4f}")
    print(f"Logit mean:    {logits_i.mean().item():.4f}")
    print(f"Logit std:     {logits_i.std().item():.4f}")
    print("-" * 35)

Image 1
Logit minimum: 0.0266
Logit maximum: 0.1047
Logit mean:    0.0419
Logit std:     0.0056
-----------------------------------
Image 2
Logit minimum: 0.0217
Logit maximum: 0.1102
Logit mean:    0.0427
Logit std:     0.0060
-----------------------------------
Image 3
Logit minimum: 0.0249
Logit maximum: 0.1087
Logit mean:    0.0435
Logit std:     0.0056
-----------------------------------
Image 4
Logit minimum: 0.0240
Logit maximum: 0.1156
Logit mean:    0.0591
Logit std:     0.0066
-----------------------------------


In [7]:
# Check the checkpoint's training information

print("Checkpoint epoch:", checkpoint["epoch"])
print("Training loss:", checkpoint["train_loss"])
print("Validation loss:", checkpoint["val_loss"])

# Check the model's final layer weights
final_weights = model.final_conv.weight.detach()

print("\nFinal convolution weights:")
print("Mean:", final_weights.mean().item())
print("Std:", final_weights.std().item())
print("Min:", final_weights.min().item())
print("Max:", final_weights.max().item())

Checkpoint epoch: 1
Training loss: 0.7128222361207008
Validation loss: 0.7157504856586456

Final convolution weights:
Mean: 0.0051952810026705265
Std: 0.07327908277511597
Min: -0.11949542909860611
Max: 0.11676061898469925


In [13]:
import importlib
import docforge.training.metrics as metrics

importlib.reload(metrics)

print("Available functions:")
print([name for name in dir(metrics) if not name.startswith("_")])

Available functions:
['calculate_metrics', 'combined_loss', 'dice_loss', 'torch']


In [14]:
from docforge.training.metrics import combined_loss

test_loss = combined_loss(outputs, masks)

print("Combined loss:", test_loss.item())

Combined loss: 1.6611671447753906
